---
technologies: "Redis"
category: "Choice and use of technology"
difficulty: "Easy"
---

# Redis

## Used material

1. <span id="used-material-1"></span> [Redis pip package](https://pypi.org/project/redis/)

2. <span id="used-material-2"></span> [Run Redis on Docker](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/docker/)

3. <span id="used-material-3"></span> [Redis python guide](https://redis.io/docs/latest/develop/clients/redis-py/)

4. <span id="used-material-4"></span> [Redis guide](https://www.datacamp.com/tutorial/python-redis-beginner-guide)

## Why use Redis?

Redis was chosen for the following reasons:

- A common cache database with a good starting point for distributed locking (mature)

- Easy to setup and use with a Python client (abstracted)

- Usable in any place that supports Docker (interoperable)

These enable us to use Redis to save parameters, move data between containers, and ensure deterministic actions.

## How to use Redis?

Use the [packages](./packages.txt) to setup your venv [(1)](#used-material-1). Assuming your Docker Compose is setup, let's store parameters into Redis to be used later. Setup the container with the [Redis](./redis.yaml) file in the following way [(2)](#used-material-2):

```
docker compose -f redis.yaml up
```

Now, we have abstracted the use of Redis |[(3)](#used-material-3), [(4)](#used-material-4)| with [setup](./redis_func/setup.py) and [use](./redis_func/use.py) functions, which we will use with [caching](./caching.py) and [locking](./locking.py) functions to store a nested dictionary in the following way:

In [9]:
%run ./caching.py
%run ./locking.py

In [10]:
cache_key = caching_generate_key(
    static = True,
    user = 'part-2',
    target = 'notebook',
    group = 'dict'
)

In [11]:
print(cache_key)

cache:part-2:notebook:dict


In [12]:
nested_example_dict = {
    'item-1': {
        'nest-1': {
            'value': 'empty'
        },
        'nest-2': {
            'value': 'empty'
        }
    },
    'item-2': {
        'nest-1': {
            'value': 'empty'
        },
        'nest-2': {
            'value': 'empty'
        }
    }
}

In [13]:
cache_response = caching_save_dict(
    cache_parameters = {
        'system': 'redis',
        'redis-endpoint': '127.0.0.1',
        'redis-port': '6379',
        'redis-db': '0'
    },
    cache_key = cache_key,
    nested_dict = nested_example_dict
)

Testing connection: 127.0.0.1:6379:0
Using key: cache:part-2:notebook:dict
redis responded with: True


In [14]:
fetched_nested_dict = caching_get_dict(
    cache_parameters = {
        'system': 'redis',
        'redis-endpoint': '127.0.0.1',
        'redis-port': '6379',
        'redis-db': '0'
    },
    cache_key = cache_key
)

Testing connection: 127.0.0.1:6379:0
Using key: cache:part-2:notebook:dict
redis gave dict: {'item-1': {'nest-1': {'value': 'empty'}, 'nest-2': {'value': 'empty'}}, 'item-2': {'nest-1': {'value': 'empty'}, 'nest-2': {'value': 'empty'}}}


We can also use the provided locking functions to make Redis handle distributed locking for us in the following way:

In [15]:
lock_client = concurrency_get_client(
    lock_parameters = {
        'system': 'redis',
        'redis-endpoint': '127.0.0.1',
        'redis-port': '6379',
        'redis-db': '0'
    }
)

lock_active, lock_name = concurrency_check_lock(
    lock_parameters = {
        'user': 'part-2',
        'target': 'notebook',
        'group': 'block',
        'resource': 'code'
    },
    lock_client = lock_client
)

if not lock_active:
    lock_created, client_lock = concurrency_get_lock(
        lock_client = lock_client,
        lock_name = lock_name
    )

    if lock_created:
        print('Locked')

    lock_released = concurrency_release_lock(
            lock_client = lock_client,
            client_lock = client_lock
        )

Testing connection: 127.0.0.1:6379:0
redis connected: True
Checking lock for lock:part-2:notebook:block:code
Lock active: False
Lock named lock:part-2:notebook:block:code active: False
Lock created: True
Locked
Lock released: True


These abstraction functions enable us to handle small data transfers between separate executions and to provide deterministic access to the same resource, provided that the executions share the same Redis instance. This requires sending the same parameters through available network connections, which we will go into detail about later.

---